## Tool Error Handling

Tools fail. APIs time out, users pass bad inputs, divisions by zero happen.
Without proper error handling:
* An unhandled exception crashes the entire agent run
* The LLM never learns *why* a tool failed and cannot recover

The key principle: **tools should never raise unhandled exceptions**.
Return a descriptive error string *or* raise `ToolException` — both give the LLM
the context it needs to respond sensibly.

**Topics covered:**
* Naive tool — what happens when it crashes
* Strategy 1: return error strings from the tool
* Strategy 2: raise `ToolException` (caught automatically by the agent)
* Comparing how the LLM responds with and without error context
* Defensive tool design checklist

In [17]:
%run langchain_common.py

Trace(trace_id=tr-a753fec2a99b19d8b51f673c49fd058d)

## 1. The problem — an unguarded tool

Let's build a tool that crashes on bad input and see what happens.

In [18]:
from pydantic import BaseModel, Field

class DivideInput(BaseModel):
    a: float = Field(description="The numerator")
    b: float = Field(description="The denominator")

@tool(args_schema=DivideInput)
def divide_naive(a: float, b: float) -> float:
    """Divide a by b."""
    return a / b  # will crash if b == 0

# Direct invocation — crashes
try:
    print(divide_naive.invoke({"a": 10, "b": 0}))
except ZeroDivisionError as e:
    print(f"Exception raised: {e}")

Exception raised: division by zero


## 2. Strategy 1 — return an error string

The simplest fix: catch the error inside the tool and return a human-readable message.
The LLM receives this string as the tool result and can explain the issue to the user.

In [19]:
@tool(args_schema=DivideInput)
def divide_safe(a: float, b: float) -> str:
    """Divide a by b. Returns an error message if b is zero."""
    if b == 0:
        return "Error: cannot divide by zero. Please provide a non-zero denominator."
    return str(a / b)

# Safe invocation
print(divide_safe.invoke({"a": 10, "b": 2}))
print(divide_safe.invoke({"a": 10, "b": 0}))

5.0
Error: cannot divide by zero. Please provide a non-zero denominator.


Trace(trace_id=tr-0d44477deea88331a4eb30f33982357d)

In [20]:
# The agent now receives the error message and can explain it to the user
agent = create_agent(llm_noreason, [divide_safe])

result = agent.invoke({"messages": [HumanMessage(content="What is 42 divided by 0?")]})
print(result["messages"][-1].content)

Division by zero is undefined in mathematics. Therefore, 42 divided by 0 has no result.


[Trace(trace_id=tr-69bd8f73deff930fef4dfe980f72a67f), Trace(trace_id=tr-39687f50a3fb4397a6de4ac1c004482e)]

## 3. Strategy 2 — raise `ToolException`

`ToolException` is LangChain's dedicated exception for tool errors.
create_agent catches it automatically and
converts it into a `ToolMessage` with `is_error=True`.
The LLM sees the error text and can decide how to proceed.

In [21]:
from langchain_core.tools import ToolException

class StockInput(BaseModel):
    ticker: str = Field(description="Stock ticker symbol, e.g. 'AAPL'")

KNOWN_STOCKS = {"AAPL": 189.50, "GOOGL": 175.30, "MSFT": 415.20}

@tool(args_schema=StockInput)
def get_stock_price(ticker: str) -> str:
    """Get the latest stock price for a given ticker symbol."""
    ticker = ticker.upper()
    if ticker not in KNOWN_STOCKS:
        raise ToolException(
            f"Ticker '{ticker}' not found. Available tickers: {', '.join(KNOWN_STOCKS)}"
        )
    return f"{ticker}: ${KNOWN_STOCKS[ticker]:.2f}"

# Invoke known ticker
print(get_stock_price.invoke({"ticker": "AAPL"}))

# Invoke unknown ticker — raises ToolException
try:
    print(get_stock_price.invoke({"ticker": "XYZ"}))
except ToolException as e:
    print(f"ToolException: {e}")

AAPL: $189.50
ToolException: Ticker 'XYZ' not found. Available tickers: AAPL, GOOGL, MSFT


Trace(trace_id=tr-3afbda4298924643ff77e17f037d17be)

In [22]:
# ToolException crashes the agent, but the agent can catch it and explain to the user

stock_agent = create_agent(llm_noreason, [get_stock_price])

queries = [
    "What is the stock price of Apple?",
    "What is the stock price of TSLA?",   # unknown — should trigger ToolException
    "What is the stock price of MSFT?"
]

for query in queries:
    try:
        result = stock_agent.invoke({"messages": [HumanMessage(content=query)]})
        print(f"Q: {query}")
        print(f"A: {result['messages'][-1].content}")
        print("-" * 60)
    except ToolException as e:
        print(f"Q: {query}")
        print(f"Error: {e}")
        print("-" * 60)

Q: What is the stock price of Apple?
A: The stock price of Apple (AAPL) is $189.50.
------------------------------------------------------------
Q: What is the stock price of TSLA?
Error: Ticker 'TSLA' not found. Available tickers: AAPL, GOOGL, MSFT
------------------------------------------------------------
Q: What is the stock price of MSFT?
A: The stock price of MSFT is $415.20.
------------------------------------------------------------


[Trace(trace_id=tr-feb4464715f13ec190fcbaee7c175eae), Trace(trace_id=tr-688cb03c658ddc6e63bec5e432acc068), Trace(trace_id=tr-e8fc0bcd7e12bd99cba016f69acec388), Trace(trace_id=tr-3d1cbfdfc3d5dd927109f02d79768f0a)]

In [23]:
# Show step-by-step message exchange for a failing tool call (ToolException path)
failing_query = "What is the stock price of TSLA?"

print(f"User query: {failing_query}\n")
print("Message exchange:\n" + "-" * 70)

try:
    for event in stock_agent.stream(
        {"messages": [HumanMessage(content=failing_query)]},
        stream_mode="values",
    ):
        msg = event["messages"][-1]
        msg_type = type(msg).__name__

        if isinstance(msg, HumanMessage):
            print(f"[HumanMessage] {msg.content}")

        elif isinstance(msg, AIMessage):
            print(f"[AIMessage] {msg.content if msg.content else '(tool call requested)'}")
            if getattr(msg, "tool_calls", None):
                print(f"  tool_calls: {msg.tool_calls}")

        elif isinstance(msg, ToolMessage):
            status = getattr(msg, "status", "unknown")
            print(f"[ToolMessage | status={status} | tool={msg.name}] {msg.content}")

        else:
            print(f"[{msg_type}] {getattr(msg, 'content', msg)}")

        print("-" * 70)

except Exception as e:
    print(f"Agent raised {type(e).__name__}: {e}")

User query: What is the stock price of TSLA?

Message exchange:
----------------------------------------------------------------------
[HumanMessage] What is the stock price of TSLA?
----------------------------------------------------------------------
[AIMessage] (tool call requested)
  tool_calls: [{'name': 'get_stock_price', 'args': {'ticker': 'TSLA'}, 'id': 'call_16116c23-e171-4f28-86b4-19929f3f1b16', 'type': 'tool_call'}]
----------------------------------------------------------------------
Agent raised ToolException: Ticker 'TSLA' not found. Available tickers: AAPL, GOOGL, MSFT


Trace(trace_id=tr-adb9442bdc42090177e0347f007f5a3a)

## 4. Handling multiple failure modes

Real tools often have several things that can go wrong.
Use specific error messages so the LLM can give targeted advice to the user.

In [24]:
class BookingInput(BaseModel):
    hotel_id: int = Field(description="Hotel ID (integer)")
    nights: int = Field(description="Number of nights (1–30)")
    promo_code: str = Field(default="", description="Optional promo code")

VALID_HOTELS = {1: "Grand Plaza", 2: "Sea View Inn", 3: "Mountain Lodge"}
VALID_PROMOS = {"SAVE10": 0.10, "SUMMER20": 0.20}

@tool(args_schema=BookingInput)
def book_hotel(hotel_id: int, nights: int, promo_code: str = "") -> str:
    """Book a hotel room for a given number of nights."""
    if hotel_id not in VALID_HOTELS:
        raise ToolException(f"Hotel ID {hotel_id} does not exist. Valid IDs: {list(VALID_HOTELS)}")
    if not (1 <= nights <= 30):
        raise ToolException(f"Nights must be between 1 and 30. You requested: {nights}")
    
    base_price = 120 * nights
    discount = 0
    if promo_code:
        if promo_code.upper() not in VALID_PROMOS:
            # Non-fatal — inform the LLM but still proceed
            return (
                f"Booking confirmed at {VALID_HOTELS[hotel_id]} for {nights} nights. "
                f"Note: promo code '{promo_code}' is invalid — full price ${base_price} charged."
            )
        discount = VALID_PROMOS[promo_code.upper()] * base_price

    final_price = base_price - discount
    return f"Booking confirmed at {VALID_HOTELS[hotel_id]} for {nights} nights. Total: ${final_price:.2f}"


booking_agent = create_agent(llm_noreason, [book_hotel])

test_cases = [
    "Book hotel 2 for 3 nights with promo code SAVE10.",
    "Book hotel 99 for 2 nights.",         # invalid hotel
    "Book hotel 1 for 50 nights.",          # too many nights
    "Book hotel 3 for 2 nights with code BADCODE.",  # invalid promo (non-fatal)
]

for q in test_cases:
    try:
        result = booking_agent.invoke({"messages": [HumanMessage(content=q)]})
        print(f"Q: {q}")
        print(f"A: {result['messages'][-1].content}")
        print("-" * 60)
    except ToolException as e:
        print(f"Q: {q}")
        print(f"Error: {e}")
        print("-" * 60)

Q: Book hotel 2 for 3 nights with promo code SAVE10.
A: Your booking at Sea View Inn for 3 nights has been confirmed. The total cost is $324.00.
------------------------------------------------------------
Q: Book hotel 99 for 2 nights.
Error: Hotel ID 99 does not exist. Valid IDs: [1, 2, 3]
------------------------------------------------------------
Q: Book hotel 1 for 50 nights.
A: I cannot book a hotel for 50 nights as the maximum allowed duration is 30 nights. Please specify a number of nights between 1 and 30.
------------------------------------------------------------
Q: Book hotel 3 for 2 nights with code BADCODE.
A: Your booking at Mountain Lodge for 2 nights is confirmed. Please note that the promo code 'BADCODE' was invalid, so the full price of $240 was charged.
------------------------------------------------------------


[Trace(trace_id=tr-aa338636b86dc0e40868d6209d056a99), Trace(trace_id=tr-7335cecbf58d9bf67817871a0ec1e28d), Trace(trace_id=tr-84c81e2770e3fb40574341d5f63a0ff8)]

## 5. Defensive tool design — checklist

```
✅  Validate all required inputs at the top of the function
✅  Use ToolException for fatal errors (wrong ID, auth failure, not found)
✅  Return informative strings for non-fatal warnings (invalid promo, rate limit)
✅  Never let unhandled Python exceptions propagate to the agent
✅  Include context in error messages (what was received, what was expected)
✅  Keep tool functions deterministic and side-effect-free when possible
```

In [ ]:
# Template: defensive tool skeleton
from typing import Any

@tool
def safe_tool_template(input_value: str) -> str:
    """Describe what this tool does clearly — the LLM reads this."""
    # 1. Validate inputs
    if not input_value:
        raise ToolException("input_value cannot be empty.")
    
    # 2. Main logic — wrap external calls in try/except
    try:
        result = input_value.upper()  # replace with real logic
    except Exception as e:
        raise ToolException(f"Operation failed: {e}")
    
    # 3. Return a descriptive result string
    return f"Result: {result}"

print(safe_tool_template.invoke({"input_value": "hello"}))